# SDH exp_018 — hierarchical gene×A-pair / position enrichment

안전한 exp13 standalone B04를 기준으로 FE만 비교합니다. raw train/test를 합치지 않으며, vocabulary·support·log-odds·표준화는 매 outer-fold train에서만 학습합니다.

실행 순서: seed42 17-case 스크리닝 → 상위 후보 3-seed → 선택적으로 고정 3-way 모델 확인.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def find_root(start):
    for path in (start, *start.parents):
        if (path / 'data' / 'raw' / 'train.csv').exists():
            return path
    raise FileNotFoundError('data/raw/train.csv가 있는 저장소 루트를 찾지 못했습니다.')

ROOT = find_root(Path.cwd().resolve())
EXP_DIR = ROOT / 'experiments' / 'SDH' / 'exp_018_hierarchical_enrichment'
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import hierarchical_enrichment as exp

train = pd.read_csv(ROOT / 'data' / 'raw' / 'train.csv')
genes = [column for column in train.columns if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
classes = np.asarray(sorted(labels.unique()))
assert len(classes) == 26
print('root:', ROOT)
print('train:', train.shape, 'genes:', len(genes), 'classes:', len(classes))

In [ ]:
CASES = exp.case_catalog()
BASELINE_CASE = exp.SAFE_BASELINE_CASE
SEEDS = (42, 52, 62)
case_table = pd.DataFrame([vars(case) for case in CASES.values()])
print('cases:', len(CASES), 'baseline:', BASELINE_CASE)
display(case_table)

## 1. Seed 42 피처 준비

가장 오래 걸리는 파싱과 nested cross-fit 점수를 5개 outer fold에 대해 한 번 준비합니다. 17개 case가 이 결과를 공유합니다.

In [ ]:
started = time.perf_counter()
prepared_by_seed = {}
prepared_by_seed[42] = exp.prepare_seed(train, genes, CASES, seed=42, verbose=True)
print(f'seed42 preparation: {(time.perf_counter() - started) / 60:.1f} min')

for item in prepared_by_seed[42]:
    audit = item.audits[BASELINE_CASE]
    assert audit['raw_train_apply_concat'] is False
    assert audit['vocabulary_source'] == 'outer_fold_fit_only'
    assert audit['fixed_cancer_names'] is False
    assert audit['fixed_exact_hotspots'] is False
print('leakage audit assertions: PASS')

## 2. Seed 42 — 17개 FE case LR 스크리닝

모델은 모든 case에서 `lbfgs, C=0.07, max_iter=2000, balanced`로 고정됩니다.

In [ ]:
screen_results = {}
screen_rows = []
started = time.perf_counter()
for index, case_name in enumerate(CASES, start=1):
    result = exp.evaluate_case(
        prepared_by_seed[42], labels, case_name, seed=42, model_kind='multinomial'
    )
    screen_results[case_name] = result
    print(f'[{index:02d}/{len(CASES)}] {case_name}: {result.macro_f1:.6f}')

baseline_42 = screen_results[BASELINE_CASE].macro_f1
for case_name, result in screen_results.items():
    row = exp.result_row(result, baseline_42)
    row['description'] = CASES[case_name].description
    row['folds_better_than_baseline'] = int(sum(
        np.asarray(result.fold_scores)
        > np.asarray(screen_results[BASELINE_CASE].fold_scores)
    ))
    screen_rows.append(row)

screen_table = pd.DataFrame(screen_rows).sort_values(
    ['oof_macro_f1', 'folds_better_than_baseline'], ascending=False
).reset_index(drop=True)
screen_table.to_csv(RESULT_DIR / 'seed42_screen.csv', index=False)
print(f'elapsed: {(time.perf_counter() - started) / 60:.1f} min')
display(screen_table)

## 3. 3-seed 확인 후보 자동 선택

baseline과 예측이 거의 같은 중복 case는 건너뛰고 성능 상위 최대 6개를 선택합니다. seed42에서 하락했더라도 상위권이면 seed 변동성 확인을 위해 포함될 수 있습니다.

In [ ]:
TOP_K = 6
MIN_DISAGREEMENT = 0.002
CONFIRM_CASES = [BASELINE_CASE]
selected_results = [screen_results[BASELINE_CASE]]

for case_name in screen_table['case']:
    if case_name == BASELINE_CASE:
        continue
    candidate = screen_results[case_name]
    disagreement = min(
        exp.prediction_disagreement(candidate, selected)
        for selected in selected_results
    )
    if disagreement >= MIN_DISAGREEMENT:
        CONFIRM_CASES.append(case_name)
        selected_results.append(candidate)
    if len(CONFIRM_CASES) >= TOP_K + 1:
        break

# 모든 후보가 지나치게 비슷한 경우에도 상위 case를 채운다.
for case_name in screen_table['case']:
    if case_name not in CONFIRM_CASES:
        CONFIRM_CASES.append(case_name)
    if len(CONFIRM_CASES) >= TOP_K + 1:
        break

print('confirmation cases:')
for name in CONFIRM_CASES:
    print(' -', name, f'{screen_results[name].macro_f1:.6f}')

## 4. Seeds 52/62 피처 준비 및 후보 확인

자는 동안 실행하기 좋은 긴 셀입니다. seed42 결과는 다시 학습하지 않고 위 결과를 재사용합니다.

In [ ]:
CONFIRM_CATALOG = {name: CASES[name] for name in CONFIRM_CASES}
all_results = {(42, name): screen_results[name] for name in CONFIRM_CASES}
confirmation_rows = [
    exp.result_row(screen_results[name], baseline_42) for name in CONFIRM_CASES
]

for seed in (52, 62):
    started = time.perf_counter()
    prepared_by_seed[seed] = exp.prepare_seed(
        train, genes, CONFIRM_CATALOG, seed=seed, verbose=True
    )
    seed_results = {}
    for case_name in CONFIRM_CASES:
        result = exp.evaluate_case(
            prepared_by_seed[seed], labels, case_name, seed=seed, model_kind='multinomial'
        )
        seed_results[case_name] = result
        all_results[(seed, case_name)] = result
    seed_baseline = seed_results[BASELINE_CASE].macro_f1
    for case_name, result in seed_results.items():
        confirmation_rows.append(exp.result_row(result, seed_baseline))
    print(f'seed={seed} done: {(time.perf_counter() - started) / 60:.1f} min')

confirmation = pd.DataFrame(confirmation_rows)
confirmation.to_csv(RESULT_DIR / 'three_seed_confirmation.csv', index=False)
display(confirmation.sort_values(['seed', 'oof_macro_f1'], ascending=[True, False]))

In [ ]:
summary = (
    confirmation.groupby('case', as_index=False)
    .agg(
        mean_f1=('oof_macro_f1', 'mean'),
        std_f1=('oof_macro_f1', 'std'),
        min_f1=('oof_macro_f1', 'min'),
        mean_delta=('delta_vs_baseline', 'mean'),
        min_delta=('delta_vs_baseline', 'min'),
        positive_seeds=('delta_vs_baseline', lambda values: int((values > 0).sum())),
        mean_features=('feature_count_mean', 'mean'),
        warnings=('convergence_warnings', 'sum'),
    )
)
summary['pass_all_seeds'] = (summary['positive_seeds'] == 3) & (summary['mean_delta'] > 0)
summary['strong_pass'] = summary['pass_all_seeds'] & (summary['mean_delta'] >= 0.003)
summary = summary.sort_values(['mean_f1', 'min_delta'], ascending=False).reset_index(drop=True)
summary.to_csv(RESULT_DIR / 'three_seed_summary.csv', index=False)
display(summary)

passing = summary[(summary['case'] != BASELINE_CASE) & summary['pass_all_seeds']]
WINNER = BASELINE_CASE if passing.empty else passing.iloc[0]['case']
print('selected winner:', WINNER)
if WINNER == BASELINE_CASE:
    print('새 FE가 3-seed 기준을 통과하지 못했습니다. baseline을 유지합니다.')

## 5. 선택 실행 — 안전한 고정 3-way 모델 확인

가장 오래 걸리는 선택 셀입니다. baseline과 FE 승자에 동일한 `multinomial 0.55 + OVR 0.30 + LGBM 0.15`를 적용합니다. 가중치를 OOF에서 다시 선택하지 않습니다. `WINNER`가 baseline이면 한 case만 실행합니다.

In [ ]:
MODEL_CASES = list(dict.fromkeys([BASELINE_CASE, WINNER]))
model_family_results = {}
model_rows = []

for seed in SEEDS:
    for case_name in MODEL_CASES:
        print(f'[3-way] seed={seed} case={case_name}', flush=True)
        result_set = exp.evaluate_three_way(
            prepared_by_seed[seed], labels, case_name, seed=seed
        )
        model_family_results[(seed, case_name)] = result_set
        for model_name, result in result_set.items():
            model_rows.append({
                'seed': seed,
                'case': case_name,
                'model': model_name,
                'oof_macro_f1': result.macro_f1,
                'oof_accuracy': result.accuracy,
                'disagreement_vs_multinomial': (
                    0.0 if model_name == 'multinomial' else
                    exp.prediction_disagreement(result_set['multinomial'], result)
                ),
                'rescue_vs_multinomial': (
                    0.0 if model_name == 'multinomial' else
                    exp.rescue_rate(result_set['multinomial'], result, labels)
                ),
            })

model_table = pd.DataFrame(model_rows)
model_table.to_csv(RESULT_DIR / 'model_family_results.csv', index=False)
model_summary = (
    model_table.groupby(['case', 'model'], as_index=False)
    .agg(
        mean_f1=('oof_macro_f1', 'mean'),
        std_f1=('oof_macro_f1', 'std'),
        mean_disagreement=('disagreement_vs_multinomial', 'mean'),
        mean_rescue=('rescue_vs_multinomial', 'mean'),
    )
    .sort_values('mean_f1', ascending=False)
)
model_summary.to_csv(RESULT_DIR / 'model_family_summary.csv', index=False)
display(model_summary)

## 6. 선택 실행 — 앙상블 계약용 OOF 확률 저장

결과 폴더는 gitignore 대상입니다. ID, 실제 target, fold, 26개 class probability를 명시적으로 저장합니다.

In [ ]:
for (seed, case_name), result_set in model_family_results.items():
    fold_number = np.zeros(len(train), dtype=np.int8)
    for item in prepared_by_seed[seed]:
        fold_number[item.valid_index] = item.fold
    for model_name, result in result_set.items():
        frame = pd.DataFrame({
            'ID': train['ID'],
            'SUBCLASS': labels,
            'fold': fold_number,
        })
        for column, class_name in enumerate(result.classes):
            frame[f'prob__{class_name}'] = result.probability[:, column]
        path = RESULT_DIR / f'oof_{case_name}_{model_name}_seed{seed}.csv'
        frame.to_csv(path, index=False)
print('OOF files saved:', RESULT_DIR)

## 판정 가이드

- `e04` 상승: 유전자별 치환 방향이 부모 mutation-type을 넘어서는 정보가 있음.
- `e07/e08` 상승: 위치 정보는 원시 열보다 class residual 압축이 적합함.
- `e10` 상승: 두 fine 축이 상보적임.
- independent만 상승하고 residual 하락: 부모 차감이 과하거나 점수 scale이 다름.
- seed42만 상승: 희귀 token 통계의 우연 가능성이 높아 채택하지 않음.
- LR은 상승하지만 3-way 하락: FE가 모델 다양성을 줄였으므로 LR 전용 후보로 보관.